In [ ]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect("SupplyChainDB.db")
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
sql_query = """
SELECT 
    [Category Name], 
    COUNT([Order Id]) AS total_orders,
    SUM(CASE WHEN [Delivery Status] = 'Late delivery' THEN 1 ELSE 0 END) AS late_orders,
    (SUM(CASE WHEN [Delivery Status] = 'Late delivery' THEN 1 ELSE 0 END) * 100.0) / COUNT([Order Id]) AS late_ratio
FROM SupplyChainDB
GROUP BY [Category Name] 
ORDER BY late_ratio DESC
"""

pd.read_sql_query(sql_query, conn)

In [ ]:
df = pd.read_sql_query("SELECT * FROM SupplyChainDB", conn)

df['is_late'] = np.where(df['Delivery Status'] == 'Late delivery', 1, 0)

In [ ]:
shipping_analysis =df.groupby('Shipping Mode').agg(
    {'is_late': ['mean', 'count']}
).sort_values(by=('is_late', 'mean'), ascending=False)
shipping_analysis

In [ ]:
df.groupby('Order Region').agg(
    {'is_late': ['mean', 'count']}
).sort_values(by=('is_late', 'mean'), ascending=False)

In [ ]:
delivery_profit = df.groupby('Delivery Status').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Order Profit Per Order', 'sum')
).sort_values(by='Total_Profit', ascending=False)
delivery_profit

In [ ]:
suspected_fraud = df[df['Order Status'] == 'SUSPECTED_FRAUD']
fraud_analysis = suspected_fraud.groupby('Type').agg(
    Fraud_Cases=('Type', 'count'),
    Total_Lost_Money=('Sales', 'sum')
).sort_values(by='Total_Lost_Money', ascending=False)
fraud_analysis

In [ ]:
import xlsxwriter
with pd.ExcelWriter('SupplyChain_Insights.xlsx', engine='xlsxwriter') as writer:
    shipping_analysis.to_excel(writer, sheet_name='Shipping_Delays')
    delivery_profit.to_excel(writer, sheet_name='Delivery_Profits')
    fraud_analysis.to_excel(writer, sheet_name='Fraud_Detection')

conn.close()
print("Excel file 'SupplyChain_Insights.xlsx' created successfully.")